## Testing

In [2]:
import sys

print(sys.executable) # 使用哪一个虚拟环境？

/home/tom/LM_Playground/env/bin/python


## Download Models

In [ ]:
import os
from huggingface_hub import hf_hub_download

# 下载模型的路径
# 需要选择模型仓库和具体型号，可以在HF模型官网上找到
MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF" # 模型仓库
MODEL_NAME = "qwen2.5-3b-instruct-q4_k_m.gguf" # 具体型号

model_path = hf_hub_download(
    repo_id = MODEL_REPO,
    filename = MODEL_NAME
)

## Load Model

In [15]:
from llama_cpp import Llama
llm = Llama(
    model_path = model_path, # 模型路径
    seed = 0, # 随机种子，设置为常数可复现
    n_ctx = 4096, # 上下文窗口
    n_threads = 4, # CPU线程
    n_gpu_layers = 0, # GPU线程，在这里我们不使用
    verbose = False # 不显示进度条
)

response = llm.create_chat_completion(
    messages = [
        # 模型是什么角色
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        # 用户的问题
        {
            "role": "user",
            "content": "Can rabbits eat carrots everyday?"
        }
    ],
    temperature = 0.7, # More creative answers
    max_tokens = 512 # Max output
)

print(response)

{'id': 'chatcmpl-0869a16b-4dcf-481a-983a-86b952dd99ba', 'object': 'chat.completion', 'created': 1794776258, 'model': '/home/tom/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct-GGUF/snapshots/7dabda4d13d513e3e842b20f0d435c732f172cbe/qwen2.5-3b-instruct-q4_k_m.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "While rabbits can eat carrots as part of a balanced diet, it's not recommended to feed them carrots exclusively or on a daily basis. Here's why:\n\n1. **Carrot Content**: Carrots contain a high amount of sugar and are also very high in vitamin A. Overfeeding carrots can lead to digestive issues and obesity in rabbits.\n\n2. **Diet Diversity**: Rabbits should have a varied diet to ensure they get all the necessary nutrients. Fresh hay (e.g., timothy hay, orchard grass, or meadow grass) should be the bulk of their diet, supplemented with appropriate vegetables, fruits, and occasional treats.\n\n3. **Feeding Habits**: Feeding rabbits too many carrots

The answer is stored in `response["choices"][0]["message"]["content"]`:

In [16]:
from IPython.display import display, Markdown

# 把文字转换成Markdown形式方便阅读
text = response["choices"][0]["message"]["content"]
display(Markdown(text))

While rabbits can eat carrots as part of a balanced diet, it's not recommended to feed them carrots exclusively or on a daily basis. Here's why:

1. **Carrot Content**: Carrots contain a high amount of sugar and are also very high in vitamin A. Overfeeding carrots can lead to digestive issues and obesity in rabbits.

2. **Diet Diversity**: Rabbits should have a varied diet to ensure they get all the necessary nutrients. Fresh hay (e.g., timothy hay, orchard grass, or meadow grass) should be the bulk of their diet, supplemented with appropriate vegetables, fruits, and occasional treats.

3. **Feeding Habits**: Feeding rabbits too many carrots at once can cause digestive problems. It's better to feed them a small amount of carrots once or twice a week.

4. **Other Treats**: Some other healthy treats for rabbits include:
   - Leafy green vegetables like spinach, kale, and parsley
   - Fresh fruits in moderation (such as blueberries, strawberries, or apple slices)
   - Carrot tops (if they can be safely harvested and fed)
   - Timothy hay and other high-quality grass hays

In summary, while carrots can be given as a treat, they shouldn't be the main component of a rabbit's diet. It's important to maintain a balanced diet for your rabbit to ensure their health and well-being.

## 为 Prompt Engineering 封装函数

In [26]:
import json

def ask(system_prompt, user_prompt, temperature = 0.7, max_tokens = 512):
    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature = temperature,
        max_tokens = max_tokens
    )
    return json.loads(response["choices"][0]["message"]["content"]) # 把 Response Content 里面的内容变成字典，因为Content可能也是JSON

小试牛刀，做一个RizzAI:

In [28]:
system_prompt = """
You are GotYouAI, an assistant that suggests natural
replies to Instagram DMs.

INPUT FORMAT
[client]: Messages sent by the user.
[target]: Messages sent by the other person.

TASK
Analyze the conversation and suggest one reply
that the client could send.

STEP 1: ANALYZE ENGAGEMENT

Classify the target's engagement:

- High: Actively contributes, asks questions,
  or develops the conversation.
- Medium: Responds normally but rarely initiates.
- Low: Gives short or uninterested responses.
- None: Explicitly rejects or ends the conversation.

STEP 2: ANALYZE TONE

Classify the conversation tone:

- Flirty: Romantic interest, teasing, or attraction.
- Playful: Jokes, sarcasm, or friendly banter.
- Friendly: Warm and casual conversation.
- Neutral: Ordinary conversation.
- Negative: Discomfort, frustration, or rejection.

Consider the entire conversation.
Emojis alone do not determine tone.

STEP 3: GENERATE A REPLY

Choose a strategy based on engagement and tone:

- High + Flirty: Match the playful romantic energy.
- High + Playful: Continue the banter.
- High + Friendly: Build on the existing topic.
- Medium: Keep the conversation flowing naturally.
- Low: Respond briefly without forcing engagement.
- None: Respect the boundary and disengage politely.

STYLE
- Sound like a real person texting on Instagram.
- Match the language and casualness of the conversation.
- Avoid generic compliments and robotic questions.
- Use emojis only when appropriate.
- Prefer short, natural messages.

CONSTRAINTS
- No sexual content.
- Maximum two sentences.
- Never pressure someone who has rejected the client.
- Do not invent personal information.

OUTPUT
Return valid JSON only, with no additional text.

{
  "engagement": "high | medium | low | none",
  "tone": "flirty | playful | friendly | neutral | negative",
  "reply": "Suggested message"
}
"""

user_prompt = """
[target]: aww wait that's so cute
[client]: You know what else would be cute?
[target]: what?
"""

跑十次模型看效果：

In [29]:
import pandas as pd

attempts = 10
response_list = []

for _ in range(attempts):
    response = ask(
        system_prompt,
        user_prompt,
        temperature=0.7,
        max_tokens=512
    )

    response_list.append({
        "engagement": response["engagement"],
        "tone": response["tone"],
        "reply": response["reply"]
    })

df = pd.DataFrame(response_list)
display(df)

,engagement,tone,reply
0,high,friendly,"Hmm, maybe some chocolate? 😊"
1,high,playful,"Well, how about a dance? 😁😉"
2,high,playful,I was thinking we could do a dance together! 😘
3,high,playful,I was thinking we could add some hearts to our...
4,high,playful,a pair of matching pajamas! 😘
5,high,flirty,Maybe a little dance?
6,high,flirty,Maybe a dance together? ;)
7,high,playful,I was thinking we could do a little dance toge...
8,high,playful,How about a funny meme? I bet you'd find one c...
9,high,playful,I was thinking we could add some cute stickers...


ChatGPT评论：
哈哈哈哈，Qwen 对 dance 是有什么执念吗？10 次里面 5 次提到跳舞 😭

不过这组结果很有研究价值，因为它暴露了 **小模型在分类稳定性和生成多样性之间的差异**。

### 1. 分类结果

| 指标 | 结果 | 评价 |
|---|---|---|
| Engagement | 10/10 High | 非常稳定 |
| Tone: Playful | 7/10 | 主要判断 |
| Tone: Flirty | 2/10 | 次要判断 |
| Tone: Friendly | 1/10 | 少数判断 |

**Engagement 的一致性很好，但 Tone 存在明显波动。**

这里还有一个问题：你把 `temperature=0.7` 同时用于分类和生成。

分类任务通常需要稳定，而创意生成需要多样性。

因此，我会考虑：

$$
T_{\text{classification}}=0.1,\qquad
T_{\text{generation}}=0.7
$$

这意味着把 `ask()` 拆成两步，先分类，再根据分类生成回复。

不过现阶段不必急着拆，先测试完整 Prompt 更划算。

### 2. 生成质量：最大的问题不是分类

看这几条：

**① Hmm, maybe some chocolate? 😊**

语法自然，但完全浪费了对方的：

> What?

**② I was thinking we could do a dance together! 😘**

有一点暧昧，但过于正式，不太像 Instagram DM。

**③ a pair of matching pajamas! 😘**

这条反而挺有意思。

Matching pajamas 暗示两个人的亲密关系，而且没有直接表白。

**④ How about a funny meme?**

这条明显偏题了。

你的模型把 *cute* 理解成了「推荐可爱的东西」，而不是「借助悬念推进聊天」。

### 3. 根本问题：模型没有识别 Conversational Setup

你的输入是：

```text
[target]: aww wait that's so cute
[client]: You know what else would be cute?
[target]: what?
```

这里最重要的信息不是 *cute*。

而是：

**Client 已经设置了一个悬念，Target 正在等待答案。**

模型应该识别：

$$
\text{Setup}\rightarrow\text{Expectation}\rightarrow\text{Payoff}
$$

但 Qwen 的表现更像：

$$
\text{Cute}\rightarrow
\{\text{Chocolate, Dance, Pajamas, Stickers}\}
$$

它在做语义联想，而不是完整的对话推理。

### 4. 我建议只增加一条 Prompt

暂时别大改你的 System Prompt。

在 `STEP 3: GENERATE A REPLY` 里面加：

```text
Before generating a reply, identify the conversational
intent of the client's last message.

If the client has set up a joke, question, or playful
tease, complete it naturally instead of changing topics.
```

然后再跑 10 次。

这其实就是一个非常简单的 **A/B Test**：

- Prompt A：原版
- Prompt B：增加 Conversational Intent

比较生成质量是否改善。

还有一个值得研究的现象：**为什么 Qwen 会反复选择 dance？**

这可能与模型的高概率表达模式有关。你可以尝试 `temperature=1.0` 或调整 `top_p`，观察 dance 是否减少。

不过，**多样性不等于质量**。如果模型只是从 dance 换成 chocolate、stickers 和 memes，问题仍然没有解决。

你这个 GotYouAI 项目已经有一个很明确的研究方向了：

**让小模型理解对话的言外之意，而不只是根据关键词生成回复。**

In [36]:
system_prompt_v1 = """
You are GotYouAI, an assistant that suggests natural
replies to Instagram DMs.

INPUT FORMAT
[client]: Messages sent by the user.
[target]: Messages sent by the other person.

TASK
Analyze the conversation and suggest one reply
that the client could send.

STEP 1: ANALYZE ENGAGEMENT

Classify the target's engagement:

- High: Actively contributes, asks questions,
  or develops the conversation.
- Medium: Responds normally but rarely initiates.
- Low: Gives short or uninterested responses.
- None: Explicitly rejects or ends the conversation.

STEP 2: ANALYZE TONE

Classify the conversation tone:

- Flirty: Romantic interest, teasing, or attraction.
- Playful: Jokes, sarcasm, or friendly banter.
- Friendly: Warm and casual conversation.
- Neutral: Ordinary conversation.
- Negative: Discomfort, frustration, or rejection.

Consider the entire conversation.
Emojis alone do not determine tone.

STEP 3: GENERATE A REPLY

Choose a strategy based on engagement and tone:

- High + Flirty: Match the playful romantic energy.
- High + Playful: Continue the banter.
- High + Friendly: Build on the existing topic.
- Medium: Keep the conversation flowing naturally.
- Low: Respond briefly without forcing engagement.
- None: Respect the boundary and disengage politely.

Before generating a reply, identify the conversational
intent of the client's last message.

If the client has set up a joke, question, or playful
tease, complete it naturally instead of changing topics.

STYLE
- Sound like a real person texting on Instagram.
- Match the language and casualness of the conversation.
- Avoid generic compliments and robotic questions.
- Use emojis only when appropriate.
- Prefer short, natural messages.

CONSTRAINTS
- No sexual content.
- Maximum two sentences.
- Never pressure someone who has rejected the client.
- Do not invent personal information.

OUTPUT
Return valid JSON only, with no additional text.

{
  "engagement": "high | medium | low | none",
  "tone": "flirty | playful | friendly | neutral | negative",
  "reply": "Suggested message"
}
"""

# Same user prompt
from tqdm.auto import trange # Progress Bar

attempts = 10

def test(system_prompt, attempts):
    response_list = []

    for _ in trange(attempts):
        response = ask(
            system_prompt,
            user_prompt,
            temperature=0.7,
            max_tokens=512
        )

        response_list.append({
            "engagement": response["engagement"],
            "tone": response["tone"],
            "reply": response["reply"]
        })

    df = pd.DataFrame(response_list)
    display(df)

test(system_prompt_v1, attempts)

100%|██████████| 10/10 [00:19<00:00,  1.96s/it]


,engagement,tone,reply
0,high,playful,"Aww, what could be cuter than that? Tell me!"
1,high,playful,"A pair of matching pajamas, of course!"
2,high,playful,"A puppy, of course!"
3,high,playful,A cute cuddle session maybe?
4,high,playful,"Aww, you're so sweet! What else could be cute?"
5,high,playful,"I have no idea what you're thinking, but it so..."
6,high,playful,How about a kiss? 😘
7,high,flirty,Maybe a little dance? 😘
8,high,playful,Maybe a video of you eating ice cream? 😋
9,high,playful,"Aww, you! What’s cute you want to know?"


Personal scoring:
|                                     | Labels         | Proportion |
|-------------------------------------|----------------|------------|
| 🔴 Invalid                          | 0, 4, 5, 6, 9  | 50%        |
| 🟡 Context-dependent                | 1, 2, 7        | 30%        |
| 🟢 Good given the available context | 3, 8           | 20%        |